RL agent 

In [2]:
import os, random, uuid
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model, Model

from sgtlib import modules as sgt

In [ ]:
# load cnn
CNN_PATH = "graphclassifier.keras"
cnn_model = load_model(CNN_PATH)

In [ ]:
# Preprocessing
IMG_SIZE = (224, 224)

def _ensure_rgb(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 2:  # grayscale
        arr = np.repeat(arr[..., None], 3, axis=2)
    if arr.shape[-1] == 4:  # drop alpha
        arr = arr[..., :3]
    return arr

def preprocess_for_cnn(image_array: np.ndarray) -> np.ndarray:
    arr = np.asarray(image_array)
    arr = _ensure_rgb(arr)
    arr = tf.image.resize(tf.convert_to_tensor(arr, dtype=tf.float32), IMG_SIZE).numpy()
    return arr

def cnn_predict_good(graph_image: np.ndarray) -> float:
    x = preprocess_for_cnn(graph_image)
    x = np.expand_dims(x, axis=0)
    prob = cnn_model.predict(x, verbose=0)[0][0]  # binary sigmoid
    return float(prob)

# Apply StructuralGT filters
def _safe_cfg_set(cfgs: Dict[str, Any], key: str, field: str, value: Any):
    if key in cfgs and isinstance(cfgs[key], dict) and field in cfgs[key]:
        cfgs[key][field] = value

def apply_filters(img_path: str, config: Dict[str, Any]) -> Optional[np.ndarray]:
    ntwk_obj, _ = sgt.ImageProcessor.create_imp_object(str(img_path))
    cfgs = ntwk_obj.image_obj.configs

    _safe_cfg_set(cfgs, "threshold_type", "value", config.get("threshold_type", 0))
    _safe_cfg_set(cfgs, "global_threshold_value", "value", config.get("global_threshold_value", 128))
    _safe_cfg_set(cfgs, "adaptive_local_threshold_value", "value", config.get("adaptive_local_threshold_value", 11))
    _safe_cfg_set(cfgs, "otsu", "value", 1 if config.get("threshold_type", 0) == 2 else 0)

    _safe_cfg_set(cfgs, "apply_gamma", "value", config.get("apply_gamma", 0))
    _safe_cfg_set(cfgs, "apply_gamma", "dataValue", config.get("lut_gamma", 1.0))

    _safe_cfg_set(cfgs, "apply_autolevel", "value", config.get("apply_autolevel", 0))
    _safe_cfg_set(cfgs, "apply_autolevel", "dataValue", config.get("autolevel_blur_size", 3))

    _safe_cfg_set(cfgs, "apply_gaussian_blur", "value", config.get("apply_gaussian_blur", 0))
    _safe_cfg_set(cfgs, "apply_gaussian_blur", "dataValue", config.get("gaussian_blur_size", 3))

    _safe_cfg_set(cfgs, "apply_lowpass_filter", "value", config.get("apply_lowpass_filter", 0))
    _safe_cfg_set(cfgs, "apply_lowpass_filter", "dataValue", config.get("lowpass_window_size", 50))

    _safe_cfg_set(cfgs, "apply_laplacian_gradient", "value", config.get("apply_laplacian_gradient", 0))
    _safe_cfg_set(cfgs, "apply_laplacian_gradient", "dataValue", config.get("laplacian_kernel_size", 3))

    _safe_cfg_set(cfgs, "apply_sobel_gradient", "value", config.get("apply_sobel_gradient", 0))
    _safe_cfg_set(cfgs, "apply_sobel_gradient", "dataValue", config.get("sobel_kernel_size", 3))

    _safe_cfg_set(cfgs, "apply_median_filter", "value", config.get("apply_median_filter", 0))
    _safe_cfg_set(cfgs, "apply_scharr_gradient", "value", config.get("apply_scharr_gradient", 0))
    _safe_cfg_set(cfgs, "apply_dark_foreground", "value", config.get("apply_dark_foreground", 0))

    try:
        ntwk_obj.apply_img_filters()
        ntwk_obj.build_graph_network()
    except Exception:
        return None

    graph_img = getattr(ntwk_obj, "graph_image", None)
    if graph_img is None:
        return None
    return np.asarray(graph_img)

# Discrete action space 
THRESHOLD_TYPES = [0, 1, 2]          # global / adaptive / OTSU
GLOBAL_THRESH   = [96, 128, 160]     # used if tt == 0
ADAPTIVE_LOCAL  = [5, 11, 21]        # used if tt == 1
GAMMA           = [0.7, 1.0, 1.4]
GAUSS_K         = [1, 3, 5]
AUTOLEVEL_K     = [1, 3, 5]
LOWPASS_W       = [0, 50, 200]

TOGGLE_PROFILES = [
    {"apply_gaussian_blur": 1, "apply_autolevel": 0, "apply_lowpass_filter": 0, "apply_dark_foreground": 0},
    {"apply_gaussian_blur": 0, "apply_autolevel": 1, "apply_lowpass_filter": 0, "apply_dark_foreground": 0},
    {"apply_gaussian_blur": 1, "apply_autolevel": 1, "apply_lowpass_filter": 0, "apply_dark_foreground": 0},
    {"apply_gaussian_blur": 0, "apply_autolevel": 0, "apply_lowpass_filter": 1, "apply_dark_foreground": 0},
]

def _build_action_space() -> List[Dict[str, Any]]:
    actions: List[Dict[str, Any]] = []
    for tt in THRESHOLD_TYPES:
        g_vals  = GLOBAL_THRESH if tt == 0 else [128]
        al_vals = ADAPTIVE_LOCAL if tt == 1 else [11]
        for g in g_vals:
            for al in al_vals:
                for gm in GAMMA:
                    for gk in GAUSS_K:
                        for ak in AUTOLEVEL_K:
                            for lw in LOWPASS_W:
                                for toggles in TOGGLE_PROFILES:
                                    cfg = {
                                        "threshold_type": tt,
                                        "global_threshold_value": g,
                                        "adaptive_local_threshold_value": al,
                                        "lut_gamma": gm,
                                        "gaussian_blur_size": gk,
                                        "autolevel_blur_size": ak,
                                        "lowpass_window_size": lw,
                                        "laplacian_kernel_size": 3,
                                        "sobel_kernel_size": 3,
                                        "apply_gamma": 1,
                                        "apply_autolevel": toggles["apply_autolevel"],
                                        "apply_gaussian_blur": toggles["apply_gaussian_blur"],
                                        "apply_lowpass_filter": toggles["apply_lowpass_filter"],
                                        "apply_laplacian_gradient": 0,
                                        "apply_sobel_gradient": 0,
                                        "apply_median_filter": 0,
                                        "apply_scharr_gradient": 0,
                                        "apply_dark_foreground": toggles["apply_dark_foreground"],
                                    }
                                    actions.append(cfg)
    return actions


In [ ]:
# minimal environment
@dataclass
class StepResult:
    state: np.ndarray
    reward: float
    done: bool
    info: dict

class GraphFilterEnv:
    """
    Episode: choose a base image, try up to max_steps filter configs until CNN says 'good'.
    State is a 1-D dummy vector; no embeddings.
    Reward: +1 on success; -0.05 per failed try; -1 on timeout.
    """
    def __init__(self, image_paths: List[str], max_steps: int = 6,
                 success_reward: float = 1.0, step_penalty: float = -0.05, fail_penalty: float = -1.0):
        self.images = [str(p) for p in image_paths]
        self.max_steps = max_steps
        self.success_reward = success_reward
        self.step_penalty = step_penalty
        self.fail_penalty = fail_penalty
        self.action_space = _build_action_space()
        self._curr_path: Optional[str] = None
        self._step_count = 0
        self._state = np.zeros(1, dtype=np.float32)  # minimal state

    @property
    def n_actions(self) -> int:
        return len(self.action_space)

    def reset(self) -> np.ndarray:
        self._curr_path = random.choice(self.images)
        self._step_count = 0
        return self._state

    def step(self, action_idx: int) -> StepResult:
        self._step_count += 1
        cfg = self.action_space[action_idx]

        graph_img = apply_filters(self._curr_path, cfg)
        if graph_img is None:
            done = (self._step_count >= self.max_steps)
            reward = self.fail_penalty if done else self.step_penalty
            return StepResult(self._state, reward, done, {"prob_good": None, "action": cfg, "built": False})

        prob_good = cnn_predict_good(graph_img)
        if prob_good >= 0.5:
            return StepResult(self._state, self.success_reward, True,
                              {"prob_good": float(prob_good), "action": cfg, "built": True, "step": self._step_count})

        done = (self._step_count >= self.max_steps)
        reward = self.fail_penalty if done else self.step_penalty
        return StepResult(self._state, reward, done,
                          {"prob_good": float(prob_good), "action": cfg, "built": True, "step": self._step_count})

# Helper: collect images
def make_image_list(images_dir: str) -> List[str]:
    exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".gif"}
    return [str(p) for p in Path(images_dir).rglob("*") if p.suffix.lower() in exts]

ValueError: File not found: filepath=graphclassifier.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
# reinforce agent
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from typing import List, Tuple, Dict, Any

# tiny policy: state is a 1-D dummy vector, output is logits over actions
class Policy(nn.Module):
    def __init__(self, n_actions: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.ReLU(),
            nn.Linear(64, n_actions)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)  # logits [B, n_actions]
    def dist(self, state_batch: torch.Tensor):
        logits = self.forward(state_batch)
        return torch.distributions.Categorical(logits=logits)

@torch.no_grad()
def select_action(pi: Policy, state: np.ndarray, greedy: bool = False) -> int:
    st = torch.tensor(state, dtype=torch.float32).unsqueeze(0)  # [1,1]
    logits = pi(st)
    if greedy:
        return int(torch.argmax(logits, dim=1).item())
    dist = torch.distributions.Categorical(logits=logits)
    return int(dist.sample().item())

def train_reinforce(
    env,
    epochs: int = 10,
    episodes_per_epoch: int = 20,
    gamma: float = 0.95,
    lr: float = 3e-4,
    seed: int = 42,
) -> Tuple[Policy, List[Dict[str, Any]]]:
    """
    Runs REINFORCE on the GraphFilterEnv.
    Each episode = 1 base image, up to env.max_steps actions, until success/timeout.
    """
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    nA = env.n_actions
    pi = Policy(nA).to(device)
    opt = optim.Adam(pi.parameters(), lr=lr)

    history = []
    for ep in range(1, epochs + 1):
        returns_ep, steps_ep, success_ep = [], [], 0

        for _ in range(episodes_per_epoch):
            state = env.reset()  # dummy state: [0.0]
            logps, rewards = [], []
            done, steps = False, 0

            while not done and steps < env.max_steps:
                steps += 1
                st = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)  # [1,1]
                dist = pi.dist(st)
                a = dist.sample()
                logps.append(dist.log_prob(a))

                step = env.step(int(a.item()))
                rewards.append(step.reward)
                state = step.state
                done = step.done

            # Compute discounted returns G_t
            G, Gs = 0.0, []
            for r in reversed(rewards):
                G = r + gamma * G
                Gs.append(G)
            Gs.reverse()
            R = torch.tensor(Gs, dtype=torch.float32, device=device)
            if len(R) > 1:
                R = (R - R.mean()) / (R.std() + 1e-8)  # baseline (normalization)

            # Policy gradient loss
            loss = torch.stack([ -lp * Rt for lp, Rt in zip(logps, R) ]).sum()
            opt.zero_grad()
            loss.backward()
            opt.step()

            ep_return = sum(rewards)
            returns_ep.append(ep_return)
            steps_ep.append(steps)
            success_ep += int(ep_return > 0.0)  # success if got +1
            history.append({"return": ep_return, "steps": steps})

        avg_ret  = float(np.mean(returns_ep))
        avg_step = float(np.mean(steps_ep))
        succ_rate = success_ep / episodes_per_epoch
        print(f"[REINFORCE] epoch {ep}/{epochs}  avg_return={avg_ret:.3f}  "
              f"avg_steps={avg_step:.2f}  success_rate={succ_rate:.3f}")

    return pi, history

def rollout_greedy(pi: Policy, env, img_path: str, threshold: float = 0.5):
    """
    Greedy (argmax) rollout on a single image with the trained policy.
    Returns (success_bool, used_steps, last_info_dict)
    """
    # force env to start from a specific image
    env._curr_path = img_path
    env._step_count = 0
    state = env.reset()
    steps = 0
    info = {}

    while steps < env.max_steps:
        steps += 1
        a = select_action(pi, state, greedy=True)
        step = env.step(a)
        info = step.info
        if step.done:
            return (step.reward > 0), steps, info
    return False, steps, info


In [ ]:
# Collect training images
imgs = make_image_list("images")

# Build environment
env = GraphFilterEnv(imgs, max_steps=4)  # small max_steps for faster experiments

# Train policy
pi, history = train_reinforce(
    env,
    epochs=5,               # try small number first
    episodes_per_epoch=10,  # total = 50 episodes here
    gamma=0.95,
    lr=3e-4
)

# Test the trained policy on one image
test_img = random.choice(imgs)
ok, used_steps, info = rollout_greedy(pi, env, test_img)
print("SUCCESS?", ok, "steps:", used_steps)
print("last action info:", info)
